In [18]:
import json
import os
import requests

In [2]:
with open("aws-descriptions.json") as aws_descriptions_file:
    aws_descriptions = json.loads("".join(aws_descriptions_file.readlines()))
aws_descriptions[0]

{'long': 'Application Migration Service',
 'description': 'Application Migration Service minimizes time-intensive, error-prone manual processes by automatically converting your source servers from physical, virtual, and cloud infrastructure to run natively on AWS. It further simplifies your migration by allowing you to use the same automated process for a wide range of applications.'}

In [46]:
with open("aws-intro-urls.json") as aws_intro_urls_file:
    aws_intro_urls = json.loads("".join(aws_intro_urls_file.readlines()))
aws_intro_urls[0]

{'long': 'Application Migration Service',
 'intro_url': 'https://docs.aws.amazon.com/mgn/latest/ug/what-is-mgn.html'}

In [43]:
with open("aws-translations.json") as aws_translations_file:
    aws_translations = json.loads("".join(aws_translations_file.readlines()))
aws_translations[0]

{'long': 'Application Migration Service',
 'translation': 'Interstellar Relocation Engine'}

In [25]:
aws_descriptions_set = set([item["long"] for item in aws_descriptions])
aws_intro_urls_set = set([item["long"] for item in aws_intro_urls])
aws_translations_set = set([item["long"] for item in aws_translations])
aws_descriptions_set.difference(aws_intro_urls_set), aws_descriptions_set.difference(
    aws_translations_set
), aws_intro_urls_set.difference(aws_translations_set)

(set(), set(), set())

To translate the information, create a map with the short and long fields of aws-translations.json as keys and translation as the value.
Replace each key with the translation value.

Afterwards, translate the following in the shown order:
1. AWS -> OOO
2. Amazon Web Services -> Orion Outer Orbit
3. Amazon -> OOO

In [47]:
extra_translations = [
    {"long": "AWS", "translation": "OOO"},
    {"long": "Amazon Web Services", "translation": "Orion Outer Orbit"},
    {"long": "Amazon", "translation": "OOO"},
]

In [48]:
# aws_descriptions_map = {item["long"]: item["description"] for item in aws_descriptions}
aws_intro_urls_map = {item["long"]: item["intro_url"] for item in aws_intro_urls}
aws_translations_map = {item["long"]: item["translation"] for item in aws_translations}
aws_short_name_map = {
    item["long"]: item["short"] for item in aws_translations if "short" in item
}

In [49]:
service = "Augmented AI"
service_short = aws_short_name_map.get(service)
intro_url = aws_intro_urls_map[service]
intro_markdown_url = intro_url.removesuffix("html") + "md"
service_short, intro_markdown_url

('A2I',
 'https://docs.aws.amazon.com/sagemaker/latest/dg/a2i-use-augmented-ai-a2i-human-review-loops.md')

In [27]:
docs_folder = "docs"
docs_original_folder = "original"
os.makedirs(os.path.join(docs_folder, docs_original_folder), exist_ok=True)

docs_translated_folder = "translated"
os.makedirs(os.path.join(docs_folder, docs_translated_folder), exist_ok=True)

In [52]:
original_content_path = os.path.join(docs_folder, docs_original_folder, f"{service}.md")
if not os.path.exists(original_content_path):
    response = requests.get(intro_markdown_url)

    if response.status_code == 200:
        with open(original_content_path, "wb") as file:
            file.write(response.content)
else:
    print(f"{service}.md already exists. Skipping")

Augmented AI.md already exists. Skipping


In [53]:
with open(original_content_path, "r") as file:
    original_content = file.read()

In [61]:
service_no_space = service.replace(" ", "")
service_lowercase = service_no_space.lower()
service_with_dash = service.replace(" ", "-").lower()
service_short_lowercase = service_short.lower()

service_translated = aws_translations_map[service]
service_translated_no_space = service_translated.replace(" ", "")
service_translated_lowercase = service_translated_no_space.lower()
service_translated_with_dash = service_translated.replace(" ", "-").lower()

service_no_space, service_with_dash, service_lowercase, service_short_lowercase, service_translated, service_translated_no_space, service_translated_lowercase, service_translated_with_dash

('AugmentedAI',
 'augmented-ai',
 'augmentedai',
 'a2i',
 'Android Co-Pilot',
 'AndroidCo-Pilot',
 'androidco-pilot',
 'android-co-pilot')

In [71]:
translated_content = (
    original_content.replace(service, service_translated)
    .replace(service_short, service_translated)
    .replace(service_no_space, service_translated_no_space)
    .replace(service_with_dash, service_translated_with_dash)
    .replace(service_lowercase, service_translated_lowercase)
    .replace(service_short_lowercase, service_translated_lowercase)
)

In [73]:
def translate(content, original: str, translation: str) -> str:
    original_no_space = original.replace(" ", "")
    original_lowercase = original_no_space.lower()
    original_with_dash = original.replace(" ", "-").lower()

    translation_no_space = translation.replace(" ", "")
    translation_lowercase = translation_no_space.lower()
    translation_with_dash = translation.replace(" ", "-").lower()

    return (
        content.replace(original, translation)
        .replace(original_no_space, translation_no_space)
        .replace(original_with_dash, translation_with_dash)
        .replace(original_lowercase, translation_lowercase)
    )

In [74]:
for extra_translation in extra_translations:
    translated_content = translate(
        translated_content, extra_translation["long"], extra_translation["translation"]
    )

In [75]:
path = os.path.join(docs_folder, docs_translated_folder, f"{service_translated}.md")
with open(path, "w") as file:
    file.write(translated_content)